# 232 Group Project Part 3: Preprocessing & First Model Building and Evaluation

**Import Packages**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, FloatType
from pyspark.sql.functions import col, count, length,countDistinct, broadcast, min as spark_min, max as spark_max,avg, stddev, approx_count_distinct
import requests
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml.feature import CountVectorizer, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#### SDSC Expanse Environment Setup

8 total cores and 128 GB total memory, reserving 2 GB for the Spark driver. Executor instances = 8 - 1 = 7. Executor memory = (128 - 2) / 7 = 18 GB per executor. Spark is configured with 7 executor instances and 18 GB executor memory.

Calculated executer memory is ~17.7GB but we reduced it to 16 GB and allocated 2 GB memory overhead per executor to prevent memory exhaustion during heavy operations.

In [ ]:
total_cores = 8
total_memory_gb = 128
driver_memory_gb = 4   

executor_instances = total_cores - 1
raw_executor_memory = (total_memory_gb - driver_memory_gb) / executor_instances

executor_memory_gb = 16
executor_overhead_gb = 2

print(f"Executor instances = {executor_instances}")
print(f"Raw executor memory = {raw_executor_memory:.2f} GB")
print(f"Configured executor memory = {executor_memory_gb} GB (+{executor_overhead_gb} GB overhead)")

spark = (
    SparkSession.builder
    .appName("MusicBrainz")
    .config("spark.driver.memory", f"{driver_memory_gb}g")
    .config("spark.executor.instances", executor_instances)
    .config("spark.executor.memory", f"{executor_memory_gb}g")
    .config("spark.executor.memoryOverhead", f"{executor_overhead_gb}g")
    .config("spark.sql.shuffle.partitions", executor_instances * total_cores)
    .getOrCreate()
)

sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

response = requests.get(url)
executors = response.json()

spark_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spark_df['maxMemory_GB'] = (spark_df['maxMemory'] / (1024**3)).round(2)
spark_df

### 1. Complete Preprocessing Using Spark

**Define Variables**

In [ ]:
MBDUMP = "/expanse/lustre/projects/uci157/rwheaton1/mbdump"

In [ ]:
def peek(df, n=20):
    return display(df.limit(n).toPandas())

**Define Schemas**

In [ ]:
artist_schema = StructType([
    StructField("id",               IntegerType(),   True),
    StructField("gid",              StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("sort_name",        StringType(),    True),
    StructField("begin_date_year",  IntegerType(),   True),
    StructField("begin_date_month", IntegerType(),   True),
    StructField("begin_date_day",   IntegerType(),   True),
    StructField("end_date_year",    IntegerType(),   True),
    StructField("end_date_month",   IntegerType(),   True),
    StructField("end_date_day",     IntegerType(),   True),
    StructField("type",             IntegerType(),   True),
    StructField("area",             IntegerType(),   True),
    StructField("gender",           IntegerType(),   True),
    StructField("comment",          StringType(),    True),
    StructField("edits_pending",    IntegerType(),   True),
    StructField("last_updated",     StringType(),    True),
    StructField("ended",            StringType(),    True),
    StructField("begin_area",       IntegerType(),   True),
    StructField("end_area",         IntegerType(),   True),
])
instrument_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("description",   StringType(),  True),
])
label_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("gid",               StringType(),  True),
    StructField("name",              StringType(),  True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("label_code",        IntegerType(), True),
    StructField("type",              IntegerType(), True),
    StructField("area",              IntegerType(), True),
    StructField("comment",           StringType(),  True),
    StructField("edits_pending",     IntegerType(), True),
    StructField("last_updated",      StringType(),  True),
    StructField("ended",             StringType(),  True),
])
genre_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
area_schema = StructType([
    StructField("id",               IntegerType(), True),
    StructField("gid",              StringType(),  True),
    StructField("name",             StringType(),  True),
    StructField("type",             IntegerType(), True),
    StructField("edits_pending",    IntegerType(), True),
    StructField("last_updated",     StringType(),  True),
    StructField("begin_date_year",  IntegerType(), True),
    StructField("begin_date_month", IntegerType(), True),
    StructField("begin_date_day",   IntegerType(), True),
    StructField("end_date_year",    IntegerType(), True),
    StructField("end_date_month",   IntegerType(), True),
    StructField("end_date_day",     IntegerType(), True),
    StructField("ended",            StringType(),  True),
    StructField("comment",          StringType(),  True),
])
tag_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("ref_count",     IntegerType(), True),
])
gender_schema = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
])
release_group_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("title",         StringType(),  True),
    StructField("artist_credit", IntegerType(), True),
    StructField("type",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
l_artist_label_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # label
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
l_artist_release_group_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # release_group
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
label_tag_schema = StructType([
    StructField("label",        IntegerType(), True),
    StructField("tag",          IntegerType(), True),
    StructField("count",        IntegerType(), True),
    StructField("last_updated", StringType(),  True),
])
l_artist_genre_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # genre
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
l_artist_artist_schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("link",    IntegerType(), True),
    StructField("entity0", IntegerType(), True),
    StructField("entity1", IntegerType(), True),
])
release_group_tag_schema = StructType([
    StructField("release_group", IntegerType(), True),
    StructField("tag",           IntegerType(), True),
    StructField("count",         IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
artist_tag_schema = StructType([
    StructField("artist",       IntegerType(), True),
    StructField("tag",          IntegerType(), True),
    StructField("count",        IntegerType(), True),
    StructField("last_updated", StringType(),  True),
])
artist_credit_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("artist_count",  IntegerType(), True),
    StructField("ref_count",     IntegerType(), True),
    StructField("created",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("gid",           StringType(),  True),
])
artist_credit_name_schema = StructType([
    StructField("artist_credit", IntegerType(), True),
    StructField("position",      IntegerType(), True),
    StructField("artist",        IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("join_phrase",   StringType(),  True),
])
l_artist_instrument_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # instrument
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])
link_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("link_type",         IntegerType(), True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("attribute_count",   IntegerType(), True),
    StructField("created",           StringType(),  True),
    StructField("ended",             StringType(),  True),
])
link_type_schema = StructType([
    StructField("id",                  IntegerType(), True),
    StructField("parent",              IntegerType(), True),
    StructField("child_order",         IntegerType(), True),
    StructField("gid",                 StringType(),  True),
    StructField("entity_type0",        StringType(),  True),
    StructField("entity_type1",        StringType(),  True),
    StructField("name",                StringType(),  True),
    StructField("description",         StringType(),  True),
    StructField("link_phrase",         StringType(),  True),
    StructField("reverse_link_phrase", StringType(),  True),
    StructField("long_link_phrase",    StringType(),  True),
    StructField("last_updated",        StringType(),  True),
    StructField("is_deprecated",       StringType(),  True),
    StructField("has_dates",           StringType(),  True),
    StructField("attribute_count",     IntegerType(), True),
    StructField("priority",            IntegerType(), True),
])
release_schema = StructType([
    StructField("id",              IntegerType(), True),
    StructField("gid",             StringType(),  True),
    StructField("name",            StringType(),  True),
    StructField("artist_credit",   IntegerType(), True),
    StructField("release_group",   IntegerType(), True),
    StructField("status",          IntegerType(), True),
    StructField("packaging",       IntegerType(), True),
    StructField("language",        IntegerType(), True),
    StructField("script",          IntegerType(), True),
    StructField("barcode",         StringType(),  True),
    StructField("comment",         StringType(),  True),
    StructField("edits_pending",   IntegerType(), True),
    StructField("quality",         IntegerType(), True),
    StructField("last_updated",    StringType(),  True),
])
release_label_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("release",        IntegerType(), True),
    StructField("label",          IntegerType(), True),
    StructField("catalog_number", StringType(),  True),
    StructField("last_updated",   StringType(),  True),
])

schemas = {
    "artist": artist_schema,
    "instrument": instrument_schema,
    "label": label_schema,
    "genre": genre_schema,
    "area": area_schema,
    "tag": tag_schema,
    "gender": gender_schema,
    "release_group": release_group_schema,
    "l_artist_label": l_artist_label_schema,
    "l_artist_release_group": l_artist_release_group_schema,
    "label_tag": label_tag_schema,
    "l_artist_genre": l_artist_genre_schema,
    "l_artist_artist": l_artist_artist_schema,
    "release_group_tag": release_group_tag_schema,
    "artist_tag": artist_tag_schema,
    "artist_credit": artist_credit_schema,
    "artist_credit_name": artist_credit_name_schema,
    "l_artist_instrument": l_artist_instrument_schema,
    "link": link_schema,
    "link_type": link_type_schema,
    "release": release_schema,
    "release_label": release_label_schema,
}

**Table Details**

In [ ]:
dfs = {}

for table_name, schema in schemas.items():
    print(f"\n==============================")
    print(f"Loading table: {table_name}")
    print(f"==============================")

    df = (
        spark.read
        .option("sep", "\t")
        .option("nullValue", r"\N")
        .option("header", "false")
        .option("quote", "")
        .option("escape", "")
        .schema(schema)
        .csv(f"{MBDUMP}/{table_name}")
    )

    dfs[table_name] = df
    
    row_count = df.count()
    print(f"Total {table_name} rows: {row_count}")
    duplicate_count = row_count - df.dropDuplicates().count()
    print(f"Duplicate rows in {table_name}: {duplicate_count}")

    print("=== SCHEMA ===")
    df.printSchema()

    print("=== PEEK ===")
    peek(df)

**Explore and Summarize Columns**

The below section creates a profiling loop that classifies columns by their role and then summarizes them. The MusicBrainz dataset has a lot of IDs and relationship table columns so we need to sift through it and figure out which numeric columns should be summarized like a continuous variable verses an ID. The loop reports nulls, distinct counts, duplicates, categorical frequencies, numeric summaries, and foreign key uniqueness depending on the column type. 

We started by idetifying the primary keys, global ID, forgein key, date part, boolean, text descriptor and count measure clumns for each of the tables we chose.

We then created the function __get_column_role__ that classifies a given column's role/data type. This function is executed in the for loop which loops through each table's columns to summarize them based on their role which was assigned in the __get_column_role__ function. 

- ID (or id-ish) columns like id, gid, entity0, and entity1, the loop returns missing values, distinct counts, and if foreign keys match the referenced table (not sure if this is actually working how we want it to so take this part with a grain of salt). Not showing frequency distributions bc most ID values are unique and a top 10 count table wouldnt be value add

- Categorical columns are summarized using value counts and percentages to show which values are most common and whether the distribution is balanced or skewed

- Numeric/count columns are summarized with count, mean, standard deviation, min, quartiles, and max

- High-cardinality text columns, like names or UUIDs, get summarized with distinct counts and length stats instead of value counts bc there are way too many unique values

In [ ]:
primary_key_cols = {"id"}
global_id_cols = {"gid"}
foreign_key_map = {
    "artist": {
        "type":       ("artist_type", "id"),
        "area":       ("area", "id"),
        "gender":     ("gender", "id"),
        "begin_area": ("area", "id"),
        "end_area":   ("area", "id"),
    },
    "instrument": {
        "type": ("instrument_type", "id"),
    },
    "label": {
        "type": ("label_type", "id"),
        "area": ("area", "id"),
    },
    "release_group": {
        "type":          ("release_group_primary_type", "id"),
        "artist_credit": ("artist_credit", "id"),
    },
    "release": {
        "artist_credit": ("artist_credit", "id"),
        "release_group": ("release_group", "id"),
        "status":        ("release_status", "id"),
        "packaging":     ("release_packaging", "id"),
        "language":      ("language", "id"),
        "script":        ("script", "id"),
    },
    "release_label": {
        "release": ("release", "id"),
        "label":   ("label", "id"),
    },
    "artist_credit_name": {
        "artist_credit": ("artist_credit", "id"),
        "artist":        ("artist", "id"),
    },
    "label_tag": {
        "label": ("label", "id"),
        "tag":   ("tag", "id"),
    },
    "artist_tag": {
        "artist": ("artist", "id"),
        "tag":    ("tag", "id"),
    },
    "release_group_tag": {
        "release_group": ("release_group", "id"),
        "tag":           ("tag", "id"),
    },
    "l_artist_genre": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("genre", "id"),
    },
    "l_release_group_genre": {
        "link":    ("link", "id"),
        "entity0": ("release_group", "id"),
        "entity1": ("genre", "id"),
    },
    "l_artist_label": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("label", "id"),
    },
    "l_artist_release_group": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("release_group", "id"),
    },
    "l_artist_artist": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("artist", "id"),
    },
    "l_artist_instrument": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("instrument", "id"),
    },
    "link": {
        "link_type": ("link_type", "id"),
    },
    "link_type": {
        "parent": ("link_type", "id"),
    },
}
date_part_cols = {"begin_date_year", "begin_date_month", "begin_date_day", "end_date_year", "end_date_month", "end_date_day"}
boolean_like_cols = {"ended"}
text_descriptor_cols = {"name", "sort_name", "comment", "join_phrase", "last_updated"}
count_measure_cols = {"count", "ref_count", "artist_count", "edits_pending", "position"}


In [ ]:
def get_column_role(table_name, column_name, row_count, distinct_count):
    distinct_ratio = distinct_count / row_count if row_count else 0

    table_fk = foreign_key_map.get(table_name, {})

    rules = [
                (column_name in primary_key_cols, "primary_key"),
                (column_name in global_id_cols, "global_identifier"),
                (column_name in table_fk, "foreign_key"),
                (column_name in date_part_cols, "date_part"),
                (column_name in boolean_like_cols, "boolean_like"),
                (column_name in count_measure_cols, "count_measure"),
            ]

    for condition, role in rules:
        if condition:
            return role

    if column_name in text_descriptor_cols:
        return "high_cardinality_text" if distinct_ratio > 0.9 else "categorical_text"

    if distinct_ratio > 0.9:
        return "identifier_like"

    if distinct_count <= 25:
        return "categorical"

    return "numeric_or_high_cardinality"

In [ ]:
profile_rows = []

for table_name, df in dfs.items():
    print(f"\n\n==============================")
    print(f"TABLE: {table_name}")
    print(f"==============================")

    for field in df.schema.fields:
        column_name = field.name
        dtype = field.dataType.simpleString()

        missing_count = df.filter(col(column_name).isNull()).count()
        distinct_count = df.select(approx_count_distinct(col(column_name))).first()[0]
        distinct_ratio = distinct_count / row_count if row_count else 0

        role = get_column_role(table_name, column_name, row_count, distinct_count)

        print(f"\n--- {column_name} ({dtype}) ---")
        print(f"Role: {role}")
        print(f"Missing: {missing_count}")
        print(f"Distinct: {distinct_count}")
        print(f"Distinct ratio: {distinct_ratio:.4f}")

        summary = {
            "table": table_name,
            "column": column_name,
            "dtype": dtype,
            "role": role,
            "row_count": row_count,
            "missing_count": missing_count,
            "missing_pct": missing_count / row_count if row_count else None,
            "distinct_count": distinct_count,
            "distinct_ratio": distinct_ratio,
            "duplicate_rows_in_table": duplicate_count,
        }

        if role in ["primary_key", "global_identifier", "identifier_like"]:
            print("Summary: identifier column; frequency distributions are not meaningful.")

            if isinstance(field.dataType, StringType):
                df.select(
                    spark_min(length(col(column_name))).alias("min_length"),
                    spark_max(length(col(column_name))).alias("max_length")
                ).show()

        elif role == "foreign_key":
            summary["references"] = str(foreign_key_map[table_name].get(column_name))

        elif role in ["categorical", "categorical_text", "boolean_like", "date_part"]:
            print("Top values:")
            (
                df.groupBy(column_name)
                .count()
                .withColumn("pct", col("count") / row_count)
                .orderBy(col("count").desc())
                .show(10, truncate=False)
            )

        elif role in ["count_measure", "numeric_or_high_cardinality"]:
            print("Numeric summary:")
            df.select(column_name).summary(
                "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
            ).show()

        elif role == "high_cardinality_text":
            print("Summary: high-cardinality text; showing length stats instead of value counts.")
            df.select(
                spark_min(length(col(column_name))).alias("min_length"),
                spark_max(length(col(column_name))).alias("max_length"),
                avg(length(col(column_name))).alias("avg_length")
            ).show()

        profile_rows.append(summary)

profile_df = spark.createDataFrame(profile_rows)
profile_df.show(200, truncate=False)

***

**OUR APPROACH:**

After exploratory analysis and testing, we discovered an important issue with the MusicBrainz schema: directly linking artists to genres through the l_artist_genre linking table was not practical because the table is almost empty. Joining through this relationship produced a dataset of only about 400 rows, which was too small for meaningful model training.

We leveraged the richer tagging system in MusicBrainz to adress this issue. Tags and genres already have substantial semantic overlap (tags such as “rock”, “jazz”, or “metal” often function as genre labels), and using tags dramatically increased the amount of usable data. This alternative approach produced a dataset of approximately 1.5 million rows.

Using this strategy, we constructed a table containing:

- release_name
- artist_name
- area_name
- label_name
- tag_name (a list of tags associated with the release)

The first tag in the tag list was selected and treated as the target variable genre_name (since the first tag is often the most representative label for the release). The remaining tags were preserved as additional contextual features and stored as a vector under tags.

The model predicts genre_name (derived from the first tag) using features such as:

- artist name
- geographic area
- record label
- associated tags

while release_name primarily serves as a relational anchor connecting the metadata together rather than as a predictive feature itself.

Additional preprocessing and feature engineering steps were then applied:

- missing-value imputation
- filtering infrequent categories
- reducing high-cardinality categorical variables (area_name, tags, and genre labels)

These transformations reduced the problem to a 19 genre classification task.

We used several Spark ML preprocessing components:

- CountVectorizer to convert tag lists into sparse binary vectors
- StringIndexer to encode categorical variables such as genre, artist area, and label
- VectorAssembler to combine all engineered features into a single feature vector suitable for distributed machine learning models

This final dataset and preprocessing pipeline were then used to train distributed classification models in Spark ML, including Random Forest and Decision Tree classifiers.

In [ ]:
# pull in needed tables
release_df = dfs["release"].select(
    F.col("id").alias("release_id"),
    F.col("name").alias("release_name"),
    F.col("artist_credit").alias("release_artist_credit"),
    F.col("release_group").alias("release_release_group"),
)

acn_df = dfs["artist_credit_name"].select(
    F.col("artist_credit").alias("acn_artist_credit"),
    F.col("artist").alias("acn_artist_id"),
)

artist_df = dfs["artist"].select(
    F.col("id").alias("artist_id"),
    F.col("area").alias("artist_area_id"),
    F.col("gender").alias("artist_gender_id"),
)

gender_df = dfs["gender"].select(
    F.col("id").alias("gender_id"),
    F.col("name").alias("artist_gender"),
)

area_df = dfs["area"].select(
    F.col("id").alias("area_id"),
    F.col("name").alias("area_name"),
)

rgt_df = dfs["release_group_tag"].select(
    F.col("release_group").alias("rgt_release_group"),
    F.col("tag").alias("rgt_tag_id"),
    F.col("count").alias("tag_count"),
)

tag_df = dfs["tag"].select(
    F.col("id").alias("tag_id"),
    F.col("name").alias("tag_name"),
)

In [ ]:
# build table (release name, artist gender, area name, tag name, tag count)
model_df = (
    release_df
    .join(acn_df, release_df["release_artist_credit"] == acn_df["acn_artist_credit"], "inner")
    .join(artist_df, acn_df["acn_artist_id"] == artist_df["artist_id"], "inner")
    .join(gender_df, artist_df["artist_gender_id"] == gender_df["gender_id"], "left")
    .join(area_df, artist_df["artist_area_id"] == area_df["area_id"], "left")
    .join(rgt_df, release_df["release_release_group"] == rgt_df["rgt_release_group"], "left")
    .join(tag_df, rgt_df["rgt_tag_id"] == tag_df["tag_id"], "left")
    .select("release_name", "artist_gender", "area_name", "tag_name", "tag_count")
)

In [ ]:
# ranking tags
tag_window = Window.partitionBy("release_name").orderBy(F.col("tag_count").desc_nulls_last())
ranked_df = model_df.withColumn("tag_rank", F.row_number().over(tag_window))

# aggregating on release_name, assigning top tag to "genre_name", storing the rest in "tags"
aggregated_df = (
    ranked_df
    .groupBy("release_name")
    .agg(
        F.first("artist_gender", ignorenulls=True).alias("artist_gender"),
        F.first("area_name", ignorenulls=True).alias("area_name"),
        F.max(F.when(F.col("tag_rank") == 1, F.col("tag_name"))).alias("genre_name"),
        F.collect_set(F.when(F.col("tag_rank") > 1, F.col("tag_name"))).alias("tags"),
    ).filter(F.col("genre_name").isNotNull())
)

# filter out garbage tags and genre names (using helper real_music_genres.csv file)
valid_genres_pd = pd.read_csv("real_music_genres.csv")
valid_genres = set(valid_genres_pd["Genre"].dropna().str.strip().str.lower())
aggregated_df = aggregated_df.filter(F.lower(F.col("genre_name")).isin(list(valid_genres)))
valid_genres_bc = spark.sparkContext.broadcast(valid_genres)
filter_tags_udf = F.udf(
    lambda tags: [t for t in (tags or []) if t and t.lower() in valid_genres_bc.value], ArrayType(StringType())
)
aggregated_df = aggregated_df.withColumn("tags", filter_tags_udf(F.col("tags")))

# map each genre_name to one of 19 broad buckets (using helper genre_buckets_wide.csv file)
buckets_pd = pd.read_csv("genre_buckets_wide.csv")
genre_to_bucket = {}
for bucket in buckets_pd.columns:
    for genre in buckets_pd[bucket].dropna():
        genre_clean = genre.strip().lower()
        if genre_clean:
            genre_to_bucket[genre_clean] = bucket

bucket_lookup_df = spark.createDataFrame([(k, v) for k, v in genre_to_bucket.items()], ["genre_key", "bucket_name"])
aggregated_df = (
    aggregated_df
    .join(F.broadcast(bucket_lookup_df),
          F.lower(F.col("genre_name")) == F.col("genre_key"), "left")
    .drop("genre_name", "genre_key")
    .withColumnRenamed("bucket_name", "genre_name")
    .filter(F.col("genre_name").isNotNull())
)

# randomly impute 75% of missing values in artist_gender as 'Male', rest as 'Female'
aggregated_df = aggregated_df.withColumn(
    "artist_gender",
    F.when(F.col("artist_gender").isNull(),
        F.when(F.rand(seed=42) < 0.75, F.lit("Male")).otherwise(F.lit("Female"))
    ).otherwise(F.col("artist_gender"))
)

# keep only the top 120 most common areas and drop everything else
top_areas = (aggregated_df.groupBy("area_name").count().orderBy(F.col("count").desc())
    .limit(120).select("area_name").rdd.flatMap(lambda r: [r[0]]).collect())
aggregated_df = aggregated_df.filter(F.col("area_name").isin(top_areas))

**Final dataframe BEFORE encoding:**

In [ ]:
aggregated_df.printSchema()
aggregated_df = aggregated_df.cache()
print("Row count:", aggregated_df.count())
peek(aggregated_df)

**Final dataframe AFTER encoding:**

In [ ]:
# using CountVectorizer to convert tag list into a sparse binary vector
cv = CountVectorizer(inputCol="tags", outputCol="tag_vector", minDF=2.0)
cv_model = cv.fit(aggregated_df)
vectorized_df = cv_model.transform(aggregated_df)

# using StringIndexer to convert genre/gender/area into a numeric index
genre_indexer = StringIndexer(inputCol="genre_name", outputCol="genre_index", handleInvalid="keep")
gender_indexer = StringIndexer(inputCol="artist_gender", outputCol="gender_index", handleInvalid="keep")
area_indexer = StringIndexer(inputCol="area_name", outputCol="area_index", handleInvalid="keep")

# putting all features into a single "features" vector using VectorAssembler
assembler = VectorAssembler(inputCols=["gender_index", "area_index", "tag_vector"], outputCol="features", handleInvalid="keep")

# putting it all together
pipeline = Pipeline(stages=[genre_indexer, gender_indexer, area_indexer, assembler])
pipeline_model = pipeline.fit(vectorized_df)
final_df = pipeline_model.transform(vectorized_df)
final_df = final_df.select("release_name", "genre_index", "features").cache()

print("Tag vocabulary size:", len(cv_model.vocabulary))
genre_list = sorted([row[0] for row in aggregated_df.select("genre_name").distinct().collect()])
print("Genre count:", len(genre_list))
print("Genres:", genre_list)

print("Row count:", final_df.count())
peek(final_df)

### 2. Training Our First Distributed Model

**RF Model 1 (maxDepth = 10): Test F1 ~0.36**

In [ ]:
# train/ test split
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.cache()
test_df = test_df.cache()
print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

# inverse-frequency class weighting to deal with class imbalance
n_train = train_df.count()
n_classes = train_df.select("genre_index").distinct().count()

class_counts_df = (train_df.groupBy("genre_index").count().withColumn("class_weight",
        F.lit(float(n_train))/(F.lit(float(n_classes)) * F.col("count").cast("double"))
    ).select("genre_index", "class_weight"))

train_weighted = (train_df.join(F.broadcast(class_counts_df), on="genre_index", how="left").cache())

# build model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="genre_index",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=10,
    maxBins=128,
    seed=42,
)

rf_model = rf.fit(train_weighted)

# make predictions
train_preds = rf_model.transform(train_weighted)
test_preds = rf_model.transform(test_df)

# calculate metrics
acc_evaluator = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="f1")
train_acc = acc_evaluator.evaluate(train_preds)
test_acc = acc_evaluator.evaluate(test_preds)
train_f1 = f1_evaluator.evaluate(train_preds)
test_f1 = f1_evaluator.evaluate(test_preds)

print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'Accuracy':<20} {train_acc:>10.4f} {test_acc:>10.4f}")
print(f"{'Weighted F1':<20} {train_f1:>10.4f} {test_f1:>10.4f}")

# display per-class F1 scores
label_list = pipeline_model.stages[0].labels
per_class_evaluator = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="fMeasureByLabel")
print(f"\n{'Genre Bucket':<40} {'Test F1':>10}")
print("-" * 52)
for idx, label in enumerate(label_list):
    per_class_evaluator.setMetricLabel(float(idx))
    f1_val = per_class_evaluator.evaluate(test_preds)
    print(f"{label:<40} {f1_val:>10.4f}")

# display sample predictions with peek
print("\nSample predictions:")
peek(test_preds.select("release_name", "genre_index", "prediction"))

In [ ]:
# garbage collection
del (
    artistdf, genderdf, artists_by_gender, artist_gender,
    instrument_counts, instrument_dist,
    release_tags, release_tags_pd,
    l_artist_label, labels, label_artist_counts, label_artist_pd,
    credit_sizes, cred,
    artist_areas, art_area,
    release_df, acn_df, artist_df, gender_df, area_df, rgt_df, tag_df,
    model_df, tag_window, ranked_df, aggregated_df,
    valid_genres_pd, valid_genres, valid_genres_bc, filter_tags_udf,
    buckets_pd, genre_to_bucket, bucket_lookup_df,
    top_areas,
    cv, cv_model, vectorized_df,
    genre_indexer, gender_indexer, area_indexer, assembler, pipeline,
    genre_list,
    train_df, train_weighted,
    n_train, n_classes, class_counts_df,
    rf,
    train_preds, test_preds,
    acc_evaluator, f1_evaluator,
    train_acc, test_acc, train_f1, test_f1,
    label_list, per_class_evaluator,
    idx, label, f1_val
)
import gc; gc.collect()

**RF Model 2 (maxDepth = 15): Test F1 ~0.38**

In [ ]:
# train/ test split
train_df2, test_df2 = final_df.randomSplit([0.8, 0.2], seed=42)
train_df2 = train_df2.cache()
test_df2  = test_df2.cache()
print(f"Train rows: {train_df2.count():,}")
print(f"Test rows: {test_df2.count():,}")

# inverse-frequency class weighting to deal with class imbalance
n_train2   = train_df2.count()
n_classes2 = train_df2.select("genre_index").distinct().count()

class_counts_df2 = (train_df2.groupBy("genre_index").count().withColumn("class_weight",
        F.lit(float(n_train2)) / (F.lit(float(n_classes2)) * F.col("count").cast("double"))
    ).select("genre_index", "class_weight"))

train_weighted2 = (train_df2.join(F.broadcast(class_counts_df2), on="genre_index", how="left").cache())

# build model
rf2 = RandomForestClassifier(
    featuresCol="features",
    labelCol="genre_index",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=15,
    maxBins=128,
    seed=42,
)

rf_model2 = rf2.fit(train_weighted2)

# make predictions
train_preds2 = rf_model2.transform(train_weighted2)
test_preds2  = rf_model2.transform(test_df2)

# calculate metrics
acc_evaluator2 = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="accuracy")
f1_evaluator2 = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="f1")
train_acc2 = acc_evaluator2.evaluate(train_preds2)
test_acc2  = acc_evaluator2.evaluate(test_preds2)
train_f1_2 = f1_evaluator2.evaluate(train_preds2)
test_f1_2  = f1_evaluator2.evaluate(test_preds2)

print(f"\n{'Metric':<20} {'Train':>10} {'Test':>10}")
print("-" * 42)
print(f"{'Accuracy':<20} {train_acc2:>10.4f} {test_acc2:>10.4f}")
print(f"{'Weighted F1':<20} {train_f1_2:>10.4f} {test_f1_2:>10.4f}")

# display per-class F1 scores
label_list2 = pipeline_model.stages[0].labels
per_class_evaluator2 = MulticlassClassificationEvaluator(labelCol="genre_index", predictionCol="prediction", metricName="fMeasureByLabel")
print(f"\n{'Genre Bucket':<40} {'Test F1':>10}")
print("-" * 52)
for idx, label in enumerate(label_list2):
    per_class_evaluator2.setMetricLabel(float(idx))
    f1_val2 = per_class_evaluator2.evaluate(test_preds2)
    print(f"{label:<40} {f1_val2:>10.4f}")

# display sample predictions with peek
print("\nSample predictions:")
peek(test_preds2.select("release_name", "genre_index", "prediction"))

In [ ]:
# garbage collection
del (
    train_df2, train_weighted2,
    n_train2, n_classes2, class_counts_df2,
    rf2, train_preds2, test_preds2,
    acc_evaluator2, f1_evaluator2,
    train_acc2, test_acc2, train_f1_2, test_f1_2,
    label_list2, per_class_evaluator2,
    f1_val2,
)
import gc; gc.collect()

### 3. Fitting Analysis

We created two Random Forest models with different hyperparameters to evaluate how model complexity affected performance. Our first model used the following parameters:

- Trees = 50
- Max Depth = 10
- Max Bins = 128

This model achieved an F1 score of approximately 0.36 on the test dataset.

Our second model increased the maximum tree depth while keeping the other parameters constant:

- Trees = 50
- Max Depth = 15
- Max Bins = 128

This model achieved a slightly improved F1 score of approximately 0.38.

The improvement from increasing tree depth suggests that the original model was underfitting the data, since deeper trees were able to capture more complex relationships between the metadata and tag based features. Both models still appear to fall on the underfitting side of the fitting curve despite the improved performance. This is due to the noisy genre labels, sparse high-dimensional tag vectors, and the broad variability present in the MusicBrainz dataset whiah all contribute to the difficulty of this task.

We also experimented with additional Random Forest configurations using larger numbers of trees and greater depths. These configurations became computationally expensive because of the scale of the dataset and the size of the generated feature vectors.

The second Random Forest model performed best because the increased tree depth allowed the model to learn more detailed decision boundaries and better capture interactions between categorical metadata and tag derived features.

For Milestone 4, we plan to explore dimensionality reduction techniques such as PCA and SVD. Since the CountVectorizer step produces large sparse feature vectors, reducing dimensionality may help decrease computational cost while preserving important semantic structure within the tags. We also plan to explore clustering approaches such as K-Means in order to identify latent groupings between genres, artists, or releases that may not be captured directly through supervised classification. In addition, we may experiment with other distributed models such as Gradient Boosted Trees or XGBoost to evaluate whether boosting methods can better model the complex nonlinear relationships present in the data.


### 4. Conclusion Section

Our first model demonstrated that Random Forest classifiers can learn meaningful patterns from the MusicBrainz dataset and perform multiclass genre prediction at scale. The model achieved moderate performance, with an F1 score of approximately 0.38 across the 19 genre categories. While this indicates that the model was able to capture relationships between artist metadata, labels, geographic information, and tags, the task itself was challenging due to noisy and highly variable genre labels derived from user generated tags.

We experimented with adjusting hyperparameters, like number of trees and tree depth, in order to improve performance. However, increasing model complexity significantly increased computational cost and runtime because of the large scale of the dataset. A possible direction for improvement could be additional feature engineering and feature selection. Some tables in MusicBrainz contain rich numerical relationship data, while others contain sparse or noisy text metadata, so refining which features to include could improve performance without dramatically increasing computational requirements. Additional improvements could also come from improved genre normalization, balancing underrepresented genres, or experimenting with more advanced distributed models like Gradient Boosted Trees or XGBoost.

Distributed computing was essential for this task because the dataset contained millions of rows and high dimensional feature vectors generated from release tags. Spark allowed us to distribute preprocessing, joins, aggregations, and model training across multiple executors rather than relying on a single machine. Distributed preprocessing techniques like CountVectorizer, StringIndexer, and VectorAssembler enabled us to efficiently transform large categorical and text-based datasets into feature vectors suitable for machine learning. Without distributed computing, processing and training on a dataset of this scale would have been nearly impossible.